# SQUAREQ: Earthquake Binary Classification

This notebook compares three models for earthquake magnitude prediction (high ≥6.0 vs low <6.0):

1. **SQUAREQ**: SAC-optimized Quantum Support Vector Classifier
2. **ZZFeatureMap QSVC**: Standard Qiskit feature map
3. **Classical SVC**: RBF kernel Support Vector Classifier

## Dataset
- **Source**: `bronze.csv`
- **Target**: Binary classification (high magnitude ≥6.0)
- **Features**: 6 features selected via multi-method feature selection (MI + F-test + RF)
- **Splits**: Train (70%) / Validation (15%) / Test (15%)


In [10]:
import sys
import os
sys.path.append('../src')

import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import mutual_info_classif, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

from sac import SACAgent
from environment import FixedQMetricsQuantumEnvironment
from circuits import create_quantum_circuit_from_params


## 1. Load and Prepare Data


In [15]:
# ============================================================================
# DATA LOADING AND CLEANING
# ============================================================================
data_path = '../data/bronze.csv'
df = pd.read_csv(data_path)

# Preprocess data
essential_features = ['latitude', 'longitude', 'mag']
df_clean = df[essential_features + ['depth', 'gap', 'rms', 'magType']].dropna(subset=essential_features)

# Fill missing values
for col in ['depth', 'gap', 'rms']:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

df_clean['magType'] = df_clean['magType'].fillna('unknown')
df_clean['high_magnitude'] = (df_clean['mag'] >= 6.0).astype(int)

# Encode magType
from sklearn.preprocessing import LabelEncoder
le_magType = LabelEncoder()
df_clean['magType_encoded'] = le_magType.fit_transform(df_clean['magType'].astype(str))

# All available features
all_features = ['latitude', 'longitude', 'depth', 'gap', 'rms', 'magType_encoded']
X_all = df_clean[all_features].values
y_all = df_clean['high_magnitude'].values


# ============================================================================
# FEATURE SELECTION (Multi-method consensus voting)
# ============================================================================
n_features = 6

from sklearn.feature_selection import mutual_info_classif, f_classif
from sklearn.ensemble import RandomForestClassifier

# Method 1: Mutual Information
mi_scores = mutual_info_classif(X_all, y_all, random_state=42)
mi_indices = np.argsort(mi_scores)[-n_features:]

# Method 2: F-test
f_scores, _ = f_classif(X_all, y_all)
f_indices = np.argsort(f_scores)[-n_features:]

# Combine methods (voting)
all_indices = np.concatenate([mi_indices, f_indices])
unique, counts = np.unique(all_indices, return_counts=True)
selected_indices = unique[np.argsort(counts)[-n_features:]]
selected_features = [all_features[i] for i in selected_indices]

X_selected = X_all[:, selected_indices]
print(f"Selected features: {selected_features}")
print(f"Selected indices: {sorted(selected_indices)}")

# ============================================================================
# DATA SAMPLING AND SCALING
# ============================================================================
# Sample 500 samples for training
n_samples = 500
if n_samples and n_samples < len(X_selected):
    X_selected, _, y_all, _ = train_test_split(
        X_selected, y_all, train_size=n_samples, random_state=42, stratify=y_all
    )

# Scale features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_selected)

print(f"\nFinal dataset shape: {X_scaled.shape}")
print(f"Class distribution: {np.bincount(y_all)}")

# ============================================================================
# TRAIN/VAL/TEST SPLIT
# ============================================================================
X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled, y_all, train_size=0.7, stratify=y_all, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print(f"\n Data Splits:")
print(f"  Train: {len(y_train)} samples")
print(f"  Val:   {len(y_val)} samples")
print(f"  Test:  {len(y_test)} samples")


Selected features: ['latitude', 'longitude', 'depth', 'gap', 'rms', 'magType_encoded']
Selected indices: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Final dataset shape: (500, 6)
Class distribution: [493   7]

 Data Splits:
  Train: 350 samples
  Val:   75 samples
  Test:  75 samples


## 2. Train SAC Agent to Optimize Quantum Circuit


In [16]:
# Use subset for SAC training (500 samples)
sac_samples = min(500, len(y_train))
idx = np.random.choice(len(y_train), sac_samples, replace=False)
X_sac = X_train[idx]
y_sac = y_train[idx]

print(f"Training SAC")

# Create environment and agent
env = FixedQMetricsQuantumEnvironment(X_sac, y_sac, num_qubits=X_sac.shape[1])
agent = SACAgent(env.state_dim, env.action_dim)

# Train SAC
episodes = 100
rewards = []
best_reward = -float('inf')
best_params = None

for episode in range(episodes):
    state = env.reset()
    total_reward = 0
    
    for step in range(5):
        action = agent.select_action(state)
        next_state, reward, done = env.step(action)
        
        agent.replay_buffer.push(state, action, reward, next_state, done)
        agent.update()
        
        state = next_state
        total_reward += reward
        
        if done:
            break
    
    rewards.append(total_reward)
    
    if total_reward > best_reward:
        best_reward = total_reward
        best_params = {
            'threshold': (action[0] + 1) / 2,
            'theta_values': action[1:env.num_qubits+1],
            'gate_types': action[env.num_qubits+1:env.num_qubits*2+1],
            'entanglement_strength': action[env.num_qubits*2+1:],
            'num_qubits': env.num_qubits
        }
    
    if (episode + 1) % 20 == 0:
        print(f"  Episode {episode + 1:3d}: Reward = {total_reward:.4f}, Best = {best_reward:.4f}")

print(f"   Best reward: {best_reward:.4f}")
print(f"   Average reward: {np.mean(rewards):.4f}")


Training SAC
  Episode  20: Reward = 0.3327, Best = 0.4177
  Episode  40: Reward = 0.4008, Best = 0.4177
  Episode  60: Reward = 0.4008, Best = 0.4177
  Episode  80: Reward = 0.3817, Best = 0.4177
  Episode 100: Reward = 0.3817, Best = 0.4177
   Best reward: 0.4177
   Average reward: 0.3895


## 3. Model 1: SQUAREQ (SAC-Optimized QSVC)


In [17]:
# Create optimized quantum circuit
qc_squareq = create_quantum_circuit_from_params(
    best_params['theta_values'],
    best_params['gate_types'],
    num_qubits=best_params['num_qubits'],
    threshold=best_params['threshold'],
    entanglement_strength=best_params['entanglement_strength']
)

# Create kernel and QSVC
kernel_squareq = FidelityQuantumKernel(feature_map=qc_squareq)
qsvc_squareq = QSVC(quantum_kernel=kernel_squareq)

# Train
print("Training SQUAREQ QSVC...")
start_time = time.time()
qsvc_squareq.fit(X_train, y_train)
train_time_squareq = time.time() - start_time

# Evaluate
y_train_pred_squareq = qsvc_squareq.predict(X_train)
y_val_pred_squareq = qsvc_squareq.predict(X_val)
y_test_pred_squareq = qsvc_squareq.predict(X_test)

train_acc_squareq = accuracy_score(y_train, y_train_pred_squareq)
val_acc_squareq = accuracy_score(y_val, y_val_pred_squareq)
test_acc_squareq = accuracy_score(y_test, y_test_pred_squareq)

print(f"   Train Accuracy: {train_acc_squareq:.4f}")
print(f"   Val Accuracy:   {val_acc_squareq:.4f}")
print(f"   Test Accuracy:  {test_acc_squareq:.4f}")
print(f"   Training Time:  {train_time_squareq:.2f}s")


Training SQUAREQ QSVC...
   Train Accuracy: 0.9857
   Val Accuracy:   0.9867
   Test Accuracy:  0.9867
   Training Time:  24.66s


## 4. Model 2: ZZFeatureMap QSVC


In [18]:
# Create ZZFeatureMap
zz_feature_map = ZZFeatureMap(feature_dimension=X_train.shape[1], reps=2)
kernel_zz = FidelityQuantumKernel(feature_map=zz_feature_map)
qsvc_zz = QSVC(quantum_kernel=kernel_zz)

# Train
print("Training ZZFeatureMap QSVC...")
start_time = time.time()
qsvc_zz.fit(X_train, y_train)
train_time_zz = time.time() - start_time

# Evaluate
y_train_pred_zz = qsvc_zz.predict(X_train)
y_val_pred_zz = qsvc_zz.predict(X_val)
y_test_pred_zz = qsvc_zz.predict(X_test)

train_acc_zz = accuracy_score(y_train, y_train_pred_zz)
val_acc_zz = accuracy_score(y_val, y_val_pred_zz)
test_acc_zz = accuracy_score(y_test, y_test_pred_zz)

print(f"   Train Accuracy: {train_acc_zz:.4f}")
print(f"   Val Accuracy:   {val_acc_zz:.4f}")
print(f"   Test Accuracy:  {test_acc_zz:.4f}")
print(f"   Training Time:  {train_time_zz:.2f}s")


Training ZZFeatureMap QSVC...
   Train Accuracy: 0.9886
   Val Accuracy:   0.9867
   Test Accuracy:  0.9867
   Training Time:  178.30s


## 5. Model 3: Classical SVC (RBF Kernel)


In [19]:
# Create classical SVC
svc_classical = SVC(kernel='rbf', random_state=42)

# Train
print("Training Classical SVC (RBF)...")
start_time = time.time()
svc_classical.fit(X_train, y_train)
train_time_svc = time.time() - start_time

# Evaluate
y_train_pred_svc = svc_classical.predict(X_train)
y_val_pred_svc = svc_classical.predict(X_val)
y_test_pred_svc = svc_classical.predict(X_test)

train_acc_svc = accuracy_score(y_train, y_train_pred_svc)
val_acc_svc = accuracy_score(y_val, y_val_pred_svc)
test_acc_svc = accuracy_score(y_test, y_test_pred_svc)

print(f"   Train Accuracy: {train_acc_svc:.4f}")
print(f"   Val Accuracy:   {val_acc_svc:.4f}")
print(f"   Test Accuracy:  {test_acc_svc:.4f}")
print(f"   Training Time:  {train_time_svc:.2f}s")


Training Classical SVC (RBF)...
   Train Accuracy: 0.9857
   Val Accuracy:   0.9867
   Test Accuracy:  0.9867
   Training Time:  0.00s


## 6. Comparison and Results Summary


In [20]:
# Create comparison DataFrame
results = pd.DataFrame({
    'Model': ['SQUAREQ (SAC-optimized)', 'ZZFeatureMap QSVC', 'Classical SVC (RBF)'],
    'Train Accuracy': [train_acc_squareq, train_acc_zz, train_acc_svc],
    'Val Accuracy': [val_acc_squareq, val_acc_zz, val_acc_svc],
    'Test Accuracy': [test_acc_squareq, test_acc_zz, test_acc_svc],
    'Training Time (s)': [train_time_squareq, train_time_zz, train_time_svc]
})

print("=" * 80)
print("COMPARISON RESULTS")
print("=" * 80)
print(results.to_string(index=False))
print("=" * 80)

# Detailed metrics
print("\n\nDETAILED METRICS (Test Set):")
print("\n" + "-" * 80)
print("SQUAREQ (SAC-optimized):")
print(classification_report(y_test, y_test_pred_squareq, 
                          target_names=['Low Magnitude', 'High Magnitude']))

print("\n" + "-" * 80)
print("ZZFeatureMap QSVC:")
print(classification_report(y_test, y_test_pred_zz, 
                          target_names=['Low Magnitude', 'High Magnitude']))

print("\n" + "-" * 80)
print("Classical SVC (RBF):")
print(classification_report(y_test, y_test_pred_svc, 
                          target_names=['Low Magnitude', 'High Magnitude']))


COMPARISON RESULTS
                  Model  Train Accuracy  Val Accuracy  Test Accuracy  Training Time (s)
SQUAREQ (SAC-optimized)        0.985714      0.986667       0.986667          24.657843
      ZZFeatureMap QSVC        0.988571      0.986667       0.986667         178.297254
    Classical SVC (RBF)        0.985714      0.986667       0.986667           0.003796


DETAILED METRICS (Test Set):

--------------------------------------------------------------------------------
SQUAREQ (SAC-optimized):
                precision    recall  f1-score   support

 Low Magnitude       0.99      1.00      0.99        74
High Magnitude       0.00      0.00      0.00         1

      accuracy                           0.99        75
     macro avg       0.49      0.50      0.50        75
  weighted avg       0.97      0.99      0.98        75


--------------------------------------------------------------------------------
ZZFeatureMap QSVC:
                precision    recall  f1-score   sup

/Users/mohammadwasim/Desktop/SQUAREQ/SQUAREQ_GitHub/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/mohammadwasim/Desktop/SQUAREQ/SQUAREQ_GitHub/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/mohammadwasim/Desktop/SQUAREQ/SQUAREQ_GitHub/venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to

In [ ]:
# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
models = ['SQUAREQ', 'ZZFeatureMap', 'Classical SVC']
train_accs = [train_acc_squareq, train_acc_zz, train_acc_svc]
val_accs = [val_acc_squareq, val_acc_zz, val_acc_svc]
test_accs = [test_acc_squareq, test_acc_zz, test_acc_svc]

x = np.arange(len(models))
width = 0.25

axes[0].bar(x - width, train_accs, width, label='Train', alpha=0.8)
axes[0].bar(x, val_accs, width, label='Val', alpha=0.8)
axes[0].bar(x + width, test_accs, width, label='Test', alpha=0.8)
axes[0].set_xlabel('Model')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy Comparison')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, rotation=15, ha='right')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Runtime comparison
times = [train_time_squareq, train_time_zz, train_time_svc]
axes[1].bar(models, times, alpha=0.8, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[1].set_xlabel('Model')
axes[1].set_ylabel('Training Time (s)')
axes[1].set_title('Training Time Comparison')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()
